In [1]:
import os
print(os.getcwd())  # prints the directory Python thinks it's running from

d:\Project\CineIQ


In [2]:
import os
import pandas as pd

BASE = os.getcwd()
print(BASE)  # run this cell first to see where Jupyter is looking

ratings = pd.read_csv(os.path.join(BASE, 'ratings.csv'))
movies  = pd.read_csv(os.path.join(BASE, 'movies_metadata.csv'))

d:\Project\CineIQ


In [3]:
ratings = pd.read_csv(os.path.join(BASE, 'ratings.csv'))
movies  = pd.read_csv(os.path.join(BASE, 'movies_metadata.csv'))

print(ratings.shape)
print(movies.shape)

(25000095, 4)
(1433553, 24)


In [4]:
movies['genres']               = movies['genres'].fillna('')
movies['keywords']             = movies['keywords'].fillna('')
movies['production_companies'] = movies['production_companies'].fillna('')

movies['soup'] = (movies['genres'] + ' ' +
                  movies['keywords'] + ' ' +
                  movies['production_companies'])

# Extra safety net — drop any rows where soup is still empty
movies['soup'] = movies['soup'].fillna('')

In [5]:
from surprise import SVD, Dataset, Reader
from surprise.model_selection import cross_validate

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId','movieId','rating']], reader)

svd = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
# train on full trainset
trainset = data.build_full_trainset()
svd.fit(trainset)

def svd_scores(user_id, movie_ids):
    return {mid: svd.predict(user_id, mid).est for mid in movie_ids}

In [6]:
import pickle
with open('svd_model.pkl', 'wb') as f:
    pickle.dump(svd, f)
print("SVD saved")

SVD saved


In [7]:
import pickle

with open('svd_model.pkl', 'rb') as f:
    svd = pickle.load(f)
print("SVD loaded")

def svd_scores(user_id, movie_ids):
    return {mid: svd.predict(user_id, mid).est for mid in movie_ids}

SVD loaded


In [8]:
print("Training complete")
print(f"Number of users: {trainset.n_users}")
print(f"Number of items: {trainset.n_items}")
print(f"Number of ratings: {trainset.n_ratings}")

Training complete
Number of users: 162541
Number of items: 59047
Number of ratings: 25000095


In [9]:
links = pd.read_csv('links.csv')
print(links.shape)
print(links.head())

(62423, 3)
   movieId  imdbId   tmdbId
0        1  114709    862.0
1        2  113497   8844.0
2        3  113228  15602.0
3        4  114885  31357.0
4        5  113041  11862.0


In [10]:
# Clean tmdbId column in links
links['tmdbId'] = pd.to_numeric(links['tmdbId'], errors='coerce')
links = links.dropna(subset=['tmdbId'])
links['tmdbId'] = links['tmdbId'].astype(int)

# Clean id column in movies metadata
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
movies = movies.dropna(subset=['id'])
movies['id'] = movies['id'].astype(int)

# Merge
merged = links.merge(movies, left_on='tmdbId', right_on='id', how='inner')

print(merged.shape)
print(merged.columns.tolist())
print(merged[['movieId', 'tmdbId', 'title', 'genres']].head(10))
print(merged.shape)
print(merged.columns.tolist())

(61568, 28)
['movieId', 'imdbId', 'tmdbId', 'id', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'tagline', 'genres', 'production_companies', 'production_countries', 'spoken_languages', 'keywords', 'soup']
   movieId  tmdbId                        title  \
0        1     862                    Toy Story   
1        2    8844                      Jumanji   
2        3   15602             Grumpier Old Men   
3        4   31357            Waiting to Exhale   
4        5   11862  Father of the Bride Part II   
5        6     949                         Heat   
6        7   11860                      Sabrina   
7        8   45325                 Tom and Huck   
8        9    9091                 Sudden Death   
9       10     710                    GoldenEye   

                                 genres  
0  Animatio

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Build soup on merged, not movies
merged['soup'] = (merged['genres'].fillna('') + ' ' +
                  merged['keywords'].fillna('') + ' ' +
                  merged['production_companies'].fillna(''))

tfidf = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = tfidf.fit_transform(merged['soup'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

TF-IDF matrix shape: (61568, 10000)


In [12]:
def tfidf_scores(user_id, movie_ids, top_rated_movies):
    user_indices = []
    for mid in top_rated_movies:
        match = merged.index[merged['movieId'] == mid]
        if len(match) > 0:
            user_indices.append(match[0])
    
    if not user_indices:
        return {mid: 0.0 for mid in movie_ids}
    
    # .A converts np.matrix to np.array — fixes the TypeError
    user_profile = np.asarray(tfidf_matrix[user_indices].mean(axis=0))
    
    candidate_indices = []
    valid_ids = []
    for mid in movie_ids:
        match = merged.index[merged['movieId'] == mid]
        if len(match) > 0:
            candidate_indices.append(match[0])
            valid_ids.append(mid)
    
    if not candidate_indices:
        return {mid: 0.0 for mid in movie_ids}
    
    sims = cosine_similarity(user_profile, tfidf_matrix[candidate_indices])
    return {mid: float(sims[0][i]) for i, mid in enumerate(valid_ids)}

In [13]:
def ensemble_scores(svd_dict, tfidf_dict, w):
    all_ids = set(svd_dict) & set(tfidf_dict)
    
    svd_vals   = np.array([svd_dict[m] for m in all_ids])
    tfidf_vals = np.array([tfidf_dict[m] for m in all_ids])
    
    # Replace .ptp() with explicit max-min
    svd_norm   = (svd_vals - svd_vals.min()) / ((svd_vals.max() - svd_vals.min()) + 1e-9)
    tfidf_norm = (tfidf_vals - tfidf_vals.min()) / ((tfidf_vals.max() - tfidf_vals.min()) + 1e-9)
    
    blended = w * svd_norm + (1-w) * tfidf_norm
    scored  = dict(zip(all_ids, blended))
    return sorted(scored, key=scored.get, reverse=True)[:50]

In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
import torch

# Step 1 — Load IMDB 50K
imdb = pd.read_csv('IMDB Dataset.csv')  # column names: 'review', 'sentiment'
print(imdb.head())
print(imdb['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [15]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2)

c:\Users\Bhawesh Agrawal\anaconda3\envs\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1924.42it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSIN

In [16]:
# Step 2 — Convert labels to integers
imdb['label'] = imdb['sentiment'].map({'negative': 0, 'positive': 1})

# Step 3 — Split into train and validation
train_df, val_df = train_test_split(imdb, test_size=0.2, random_state=42)

# Step 4 — Tokenize
train_encodings = tokenizer(train_df['review'].tolist(), 
                            truncation=True, padding=True, max_length=256)
val_encodings   = tokenizer(val_df['review'].tolist(), 
                            truncation=True, padding=True, max_length=256)

# Step 5 — Build PyTorch Dataset objects
class IMDBDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) 
                for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_ds = IMDBDataset(train_encodings, train_df['label'].tolist())
val_ds   = IMDBDataset(val_encodings,   val_df['label'].tolist())

print(f"Train size: {len(train_ds)}, Val size: {len(val_ds)}")

Train size: 40000, Val size: 10000


In [ ]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments

# IMDB 50K: map stars to sentiment label (pos/neg)
# Fine-tune for 3 classes if you want: positive / mixed / negative
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    output_dir='./distilbert-sentiment',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=True)

trainer = Trainer(model=model, args=training_args,
                  train_dataset=train_ds, eval_dataset=val_ds)
trainer.train()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2316.46it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+--------------
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.248956,0.266054
2,0.147062,0.254167
3,0.056947,0.363616


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.98it/s]


TrainOutput(global_step=7500, training_loss=0.1671024792989095, metrics={'train_runtime': 717.2335, 'train_samples_per_second': 167.31, 'train_steps_per_second': 10.457, 'total_flos': 7948043919360000.0, 'train_loss': 0.1671024792989095, 'epoch': 3.0})

In [17]:
# Check which checkpoint was loaded
import os
checkpoints = os.listdir('./distilbert-sentiment')
print(checkpoints)

['checkpoint-1000', 'checkpoint-1500', 'checkpoint-2000', 'checkpoint-2500', 'checkpoint-3000', 'checkpoint-3500', 'checkpoint-4000', 'checkpoint-4500', 'checkpoint-500', 'checkpoint-5000', 'checkpoint-5500', 'checkpoint-6000', 'checkpoint-6500', 'checkpoint-7000', 'checkpoint-7500']


In [18]:
model = DistilBertForSequenceClassification.from_pretrained('./distilbert-sentiment/checkpoint-5000')
print("Best model loaded from epoch 2")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2626.73it/s]

Best model loaded from epoch 2


In [19]:
model.save_pretrained('./distilbert-sentiment-final')
tokenizer.save_pretrained('./distilbert-sentiment-final')
print("Saved")

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.29s/it]

Saved


In [ ]:
import requests
import sqlite3
import json
import time
import os
from dotenv import load_dotenv

load_dotenv()
TMDB_TOKEN = os.getenv("TMDB_TOKEN")

def get_tmdb_reviews(tmdb_id):
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}/reviews"
    headers = {"Authorization": f"Bearer {TMDB_TOKEN}"}
    params = {'language': 'en-US', 'page': 1}
    try:
        r = requests.get(url, headers=headers, params=params, timeout=5)
        results = r.json().get('results', [])
        return [rev['content'] for rev in results]
    except:
        return []

# Set up local cache database
conn = sqlite3.connect('reviews_cache.db')
conn.execute('''CREATE TABLE IF NOT EXISTS reviews 
                (tmdb_id INTEGER PRIMARY KEY, reviews_json TEXT)''')
conn.commit()

# Only fetch top 10k most rated movies
top_10k = ratings.groupby('movieId').size().nlargest(10000).index.tolist()
top_10k_tmdb = merged[merged['movieId'].isin(top_10k)]['tmdbId'].dropna().astype(int).tolist()

print(f"Fetching reviews for {len(top_10k_tmdb)} movies...")

for i, tmdb_id in enumerate(top_10k_tmdb):
    # Skip if already cached
    existing = conn.execute('SELECT 1 FROM reviews WHERE tmdb_id=?', 
                           (tmdb_id,)).fetchone()
    if existing:
        continue
    
    reviews = get_tmdb_reviews(tmdb_id)
    conn.execute('INSERT OR REPLACE INTO reviews VALUES (?,?)',
                (tmdb_id, json.dumps(reviews)))
    
    if i % 100 == 0:
        conn.commit()
        print(f"Progress: {i}/{len(top_10k_tmdb)}")
    
    time.sleep(0.05)  # rate limiting

conn.commit()
print("Done fetching reviews")

Fetching reviews for 9928 movies...
Progress: 0/9928
Progress: 100/9928
Progress: 200/9928
Progress: 300/9928
Progress: 400/9928
Progress: 500/9928
Progress: 600/9928
Progress: 700/9928
Progress: 800/9928
Progress: 900/9928
Progress: 1000/9928
Progress: 1100/9928
Progress: 1200/9928
Progress: 1300/9928
Progress: 1400/9928
Progress: 1500/9928
Progress: 1600/9928
Progress: 1700/9928
Progress: 1800/9928
Progress: 1900/9928
Progress: 2000/9928
Progress: 2100/9928
Progress: 2200/9928
Progress: 2300/9928
Progress: 2400/9928
Progress: 2500/9928
Progress: 2600/9928
Progress: 2700/9928
Progress: 2800/9928
Progress: 2900/9928
Progress: 3000/9928
Progress: 3100/9928
Progress: 3200/9928
Progress: 3300/9928
Progress: 3400/9928
Progress: 3500/9928
Progress: 3600/9928
Progress: 3700/9928
Progress: 3800/9928
Progress: 3900/9928
Progress: 4000/9928
Progress: 4100/9928
Progress: 4200/9928
Progress: 4300/9928
Progress: 4400/9928
Progress: 4500/9928
Progress: 4600/9928
Progress: 4700/9928
Progress: 4800/9

In [20]:
import sqlite3, json, pandas as pd

conn = sqlite3.connect('reviews_cache.db')
rows = conn.execute('SELECT tmdb_id, reviews_json FROM reviews').fetchall()

records = []
for tmdb_id, reviews_json in rows:
    reviews = json.loads(reviews_json)
    for review in reviews:
        records.append({'tmdbId': tmdb_id, 'review': review})

reviews_df = pd.DataFrame(records)
print(reviews_df.shape)
print(reviews_df.head())

(9902, 2)
   tmdbId                                             review
0       3  Released in 1986, Aki Kaurismaki's <i>Varjoja ...
1      11  (As I'm writing this review, Darth Vader's the...
2      11  A long time ago in a childhood not too far awa...
3      11  Star Wars (1977) is a true masterpiece of cine...
4      11  A quality start to the franchise.\r\n\r\nI say...


In [21]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch

model     = DistilBertForSequenceClassification.from_pretrained('./distilbert-sentiment-final')
tokenizer = DistilBertTokenizer.from_pretrained('./distilbert-sentiment-final')
model.eval()

# Only keep top50 movies that have at least 1 review
def rerank_with_distilbert(top50_ids, reviews_df, model, tokenizer):
    scores = {}
    for mid in top50_ids:
        movie_reviews = reviews_df[reviews_df.tmdbId == mid]['review'].tolist()[:20]
        if not movie_reviews:
            scores[mid] = 0.0  # push to bottom instead of neutral 0.5
            continue
        inputs = tokenizer(movie_reviews, return_tensors='pt',
                          truncation=True, padding=True, max_length=256)
        with torch.no_grad():
            logits = model(**inputs).logits
        probs = logits.softmax(dim=-1).mean(dim=0)
        scores[mid] = float(probs[1])
    return sorted(scores, key=scores.get, reverse=True)[:10]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 9098.65it/s]


In [22]:
def get_unrated_movies(user_id):
    rated = set(ratings[ratings.userId == user_id]['movieId'].tolist())
    all_movies = set(merged['movieId'].tolist())
    return list(all_movies - rated)

def get_top_rated(user_id, n=20):
    user_ratings = ratings[ratings.userId == user_id]
    return user_ratings.nlargest(n, 'rating')['movieId'].tolist()

# Test with user 1
user_id    = 1
unrated    = get_unrated_movies(user_id)
svd_s      = svd_scores(user_id, unrated)
tfidf_s    = tfidf_scores(user_id, unrated, get_top_rated(user_id))
top50      = ensemble_scores(svd_s, tfidf_s, w=0.8)

# Convert movieIds to tmdbIds for DistilBERT
top50_tmdb = merged[merged['movieId'].isin(top50)]['tmdbId'].tolist()
top10_tmdb = rerank_with_distilbert(top50_tmdb, reviews_df, model, tokenizer)

# Convert back to titles
top10_titles = merged[merged['tmdbId'].isin(top10_tmdb)][['title','tmdbId']]
print(top10_titles)

                           title  tmdbId
835                The Godfather     238
2252             The Celebration     309
2747             American Beauty      14
2797              Boys Don't Cry     226
7308              Happy Together   18329
8041                Garden State     401
9492      Rocco and His Brothers    8422
14395                 About Elly   37181
20173  Blue Is the Warmest Color  152584
41273                  Moonlight  376867


In [23]:
# Get top 50 from ensemble (before DistilBERT)
top50_titles = merged[merged['movieId'].isin(top50)][['title', 'movieId', 'tmdbId']]
print(f"Top 50 from ensemble (w=0.8):")
print(top50_titles.to_string())

Top 50 from ensemble (w=0.8):
                                                  title  movieId  tmdbId
49                                   The Usual Suspects       50     629
289                              Léon: The Professional      293     101
835                                       The Godfather      858     238
1204                                          Manhattan     1244     696
1207                                       The Graduate     1247   37247
1394                                       Guantanamera     1444   43775
1797                                 Children of Heaven     1900   21334
2252                                    The Celebration     2360     309
2477                        Lovers of the Arctic Circle     2585    1414
2622                                      Jules and Jim     2732    1628
2747                                    American Beauty     2858      14
2797                                     Boys Don't Cry     2908     226
3048                 

In [24]:
# Check how many reviews each top 50 movie has in cache
for mid in top50[:10]:
    tmdb_id = merged[merged.movieId == mid]['tmdbId'].iloc[0]
    count = len(reviews_df[reviews_df.tmdbId == tmdb_id])
    title = merged[merged.movieId == mid]['title'].iloc[0]
    print(f"{title}: {count} reviews")

American Beauty: 3 reviews
We All Loved Each Other So Much: 0 reviews
Call Me by Your Name: 0 reviews
The Celebration: 1 reviews
The Godfather: 6 reviews
Kanal: 0 reviews
Vizontele: 0 reviews
Lady Bird: 9 reviews
Amores Perros: 1 reviews
Louis C.K.: Hilarious: 0 reviews


In [27]:
import re
import nltk
from nltk.corpus import stopwords
from lime.lime_text import LimeTextExplainer
import torch

# Download stopwords once
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

explainer = LimeTextExplainer(class_names=['negative', 'positive'])

def generate_human_readable_explanation(tmdb_id, reviews_df, merged_df, model, tokenizer):
    """
    Combines Content/Rule-based metadata with LIME-extracted sentiment 
    to create a human-readable recommendation template.
    """
    # --- 1. Rule-Based Part (Genres/Metadata) ---
    movie_row = merged_df[merged_df.tmdbId == tmdb_id]
    if movie_row.empty:
        return "Recommended based on your general collaborative filtering profile."

    title = movie_row['title'].iloc[0]
    
    # Extract the primary genre safely
    genres_raw = str(movie_row['genres'].iloc[0])
    genres_list = [g.strip() for g in genres_raw.split(',') if g.strip()]
    primary_genre = f"'{genres_list[0]}'" if genres_list else "similar"

    # --- 2. LIME Explainability Part (Sentiment) ---
    movie_reviews = reviews_df[reviews_df.tmdbId == tmdb_id]['review'].tolist()
    lime_keywords = []

    if movie_reviews:
        # Take a representative review
        review = movie_reviews[0] 

        def predict_fn(texts):
            inputs = tokenizer(texts, return_tensors='pt', truncation=True, padding=True, max_length=256)
            with torch.no_grad():
                logits = model(**inputs).logits
            return logits.softmax(dim=-1).detach().numpy()

        # Extract features
        exp = explainer.explain_instance(review, predict_fn, num_features=15)

        # Filter out stopwords, punctuation, and weak weights
        for word, weight in exp.as_list():
            clean_word = re.sub(r'[^a-zA-Z]', '', word).lower()
            if weight > 0.01 and clean_word and clean_word not in stop_words:
                lime_keywords.append(clean_word)

    # --- 3. Final Template Assembly ---
    if lime_keywords:
        # Get top 3 unique sentiment words
        unique_keywords = list(dict.fromkeys(lime_keywords))[:3]
        sentiment_str = ", ".join(unique_keywords)
        
        reason = f"**{title}** is recommended because it aligns with your interest in {primary_genre} films. Furthermore, audiences praised it specifically for being '{sentiment_str}'."
    else:
        reason = f"**{title}** is recommended because it is a highly-rated {primary_genre} film that closely matches your historical viewing patterns."

    return reason

# Test the new Explainability Layer on the top recommendation
test_tmdb_id = top10_tmdb[0]
explanation = generate_human_readable_explanation(test_tmdb_id, reviews_df, merged, model, tokenizer)

print("--- EXPLAINABILITY LAYER OUTPUT ---")
print(explanation)

--- EXPLAINABILITY LAYER OUTPUT ---
**About Elly** is recommended because it aligns with your interest in 'Drama' films. Furthermore, audiences praised it specifically for being 'great, suspense, kept'.


In [28]:
import mlflow

with mlflow.start_run(run_name="cineiq_ensemble"):
    mlflow.log_param("svd_factors", 100)
    mlflow.log_param("ensemble_weight_w", 0.8)
    mlflow.log_param("tfidf_max_features", 10000)
    
    mlflow.log_metric("top10_coverage_with_reviews", 
                      sum(1 for t in top10_tmdb if len(reviews_df[reviews_df.tmdbId==t]) > 0))
    
    mlflow.sklearn.log_model(tfidf, "tfidf_vectorizer")
    print("Run logged to MLflow")

2026/06/14 12:14:11 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/06/14 12:14:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instea

Run logged to MLflow
